#### Flow Log Anomaly Detection With DBSCAN
Part of the DUNE project (https://github.com/opendr-io/dune). Clusters VPC flow-log sessions on what they did -- bytes, packets, duration, bytes per packet -- plus their direction and action, and flags the sessions that fit no cluster (DBSCAN's noise): behavior unlike the bulk of the traffic. Visual plots and pivot tables are rendered at the end for hunting exfil and / or lateral movement.


In [ ]:
# 1 - settings
PARQUET_PATH = "vflowsAgg-637602092272-november-2026.parquet"
ACTIONS      = ["ACCEPT", "REJECT"]     # sessions to cluster, e.g. ["ACCEPT"] for exfil hunting

# Numeric features, all log10 so a few huge sessions don't squash the rest. Comment a line out
# to drop one. duration = end - start (seconds); bytes_per_packet: ~40 = empty packets (probes,
# bare ACKs), ~1,500 = full packets (bulk transfer).
NUMERIC = [
    "bytes",
    "packets",
    "duration",
    "bytes_per_packet",
]
# 0/1 features: one column per value of each field (tag-direction: north-south, east-west, ...)
FLAGS = [
    "tag-direction",
    "action",
]
# Weight per feature in the distance. Numeric features are scaled to one standard deviation per
# unit (missing = 1.0); a flag field is weighted so two different values are WEIGHT apart. Keep
# flag weights above EPS (cell 6 prints it): below it DBSCAN bridges the values, e.g. puts
# ACCEPT and REJECT sessions in one cluster. Missing flag weight = FLAG_WEIGHT.
FLAG_WEIGHT = 3.0
WEIGHTS = {}

# Distinct rows: log values are rounded to ROUND (0.1 = steps of about 26%) and DBSCAN runs on
# the distinct rows, each weighted by its number of sessions -- seconds, not hours, and no
# memory blow-up from hundreds of thousands of identical sessions.
ROUND = 0.1

# DBSCAN: a row is a core point when MIN_SESSIONS sessions (its own included) lie within EPS.
# EPS = None picks it from the core-distance plot (cell 6): the EPS_PCT percentile of the
# distinct rows' core distances. Set a number to choose it yourself.
MIN_SESSIONS = 10
EPS          = None
EPS_PCT      = 95


In [ ]:
# 2 - imports
from IPython.display import display
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pyarrow as pa
import pyarrow.compute as pc
import re, textwrap
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


In [ ]:
# 3 - load: drop NODATA / SKIPDATA rows (no addresses or traffic), keep ACTIONS, build session_id.
# raw_dbscan is an untouched copy of the loaded sessions: every field as it came from the
# parquet, plus session_id. Cell 10 builds the output from it, as pyod-vflows and kmeans-mark-3 do.
flows = pd.read_parquet(PARQUET_PATH)
if "log-status" in flows.columns:
    _empty = flows["log-status"].isin(["NODATA", "SKIPDATA"])
    print(f"dropped {int(_empty.sum()):,} NODATA / SKIPDATA rows")
    flows = flows[~_empty]
_keep = flows["action"].isin(ACTIONS)
print(f"kept {int(_keep.sum()):,} {' / '.join(ACTIONS)} sessions, left out {int((~_keep).sum()):,}")
flows = flows[_keep].reset_index(drop=True)

# session_id: the session key joined with "|" -- same format as pyod-vflows' cell 3, so the same
# session has the same ID in every notebook. Don't change the field order, separator or nulls.
SESSION_KEY = ["account-id", "interface-id", "srcaddr", "srcport", "dstaddr", "dstport",
               "protocol", "start", "action"]
if _miss := [k for k in SESSION_KEY if k not in flows.columns]:
    raise KeyError(f"can't build session_id -- not in the parquet: {_miss}")
_parts = []
for _k in SESSION_KEY:
    _s = flows[_k]
    if pd.api.types.is_numeric_dtype(_s) and not pd.api.types.is_bool_dtype(_s):
        _s = _s.astype("Int64")
    _parts.append(pc.fill_null(pc.cast(pa.array(_s, from_pandas=True), pa.string()), ""))
flows["session_id"] = pd.Series(pd.arrays.ArrowStringArray(pc.binary_join_element_wise(*_parts, "|")),
                                index=flows.index)
raw_dbscan = flows.copy()
print(f"{len(flows):,} sessions")


In [ ]:
# 4 - features: log10 numerics rounded to ROUND, and 0/1 flag columns
_bytes   = flows["bytes"].astype(float).fillna(0)
_packets = flows["packets"].astype(float).fillna(0)
_raw = {
    "bytes":            _bytes,
    "packets":          _packets,
    "duration":         (flows["end"] - flows["start"]).astype(float).clip(lower=0).fillna(0),
    "bytes_per_packet": (_bytes / _packets.where(_packets > 0)).fillna(0),
}
if _bad := [f for f in NUMERIC if f not in _raw]:
    raise KeyError(f"unknown NUMERIC feature(s): {_bad} -- known: {list(_raw)}")
feat = pd.DataFrame({f"log_{f}": (np.log10(_raw[f] + 1) / ROUND).round() * ROUND for f in NUMERIC})
NUM_COLS = list(feat.columns)

FLAG_COLS = {}                                   # field -> its 0/1 columns
for _f in FLAGS:
    _v = flows[_f].astype("string").fillna("(none)")
    FLAG_COLS[_f] = []
    for _val in sorted(_v.unique()):
        _c = f"{_f}={_val}"
        feat[_c] = (_v == _val).astype(np.int8)
        FLAG_COLS[_f].append(_c)
    print(f"{_f}: " + ", ".join(f"{c.split('=', 1)[1]} {int(feat[c].sum()):,}" for c in FLAG_COLS[_f]))
print(f"{len(feat.columns)} feature columns: {', '.join(feat.columns)}")


In [ ]:
# 5 - distinct rows, weighted: DBSCAN runs on each distinct feature row once, with its number of
# sessions as sample_weight; flows_row[i] = distinct row of session i
_U, flows_row, row_sessions = np.unique(feat.to_numpy(float), axis=0, return_inverse=True,
                                        return_counts=True)
flows_row = flows_row.reshape(-1)
rows = pd.DataFrame(_U, columns=feat.columns)
rows["sessions"] = row_sessions

# scale: numerics to unit standard deviation (over sessions, via the weights); a flag field's
# columns times its weight / sqrt(2), so two different values of the field are WEIGHT apart
X = np.empty((len(rows), len(feat.columns)))
_sc = StandardScaler().fit(rows[NUM_COLS], sample_weight=row_sessions)
X[:, :len(NUM_COLS)] = _sc.transform(rows[NUM_COLS]) * [WEIGHTS.get(c[4:], 1.0) for c in NUM_COLS]
_j = len(NUM_COLS)
for _f, _cols in FLAG_COLS.items():
    X[:, _j:_j + len(_cols)] = rows[_cols].to_numpy(float) * WEIGHTS.get(_f, FLAG_WEIGHT) / np.sqrt(2)
    _j += len(_cols)
print(f"{len(rows):,} distinct rows for {len(flows):,} sessions; sessions per row: median "
      f"{int(np.median(row_sessions)):,}, max {int(row_sessions.max()):,}; "
      f"{int((row_sessions < MIN_SESSIONS).sum()):,} rows have fewer than MIN_SESSIONS ({MIN_SESSIONS})")


In [ ]:
# 6 - choose EPS: each distinct row's core distance -- how far you must go from it to take in
# MIN_SESSIONS sessions (its own count included, so a row with that many sessions is at 0).
# EPS cuts this curve: rows above it are noise unless they sit within EPS of a core row.
_k = min(len(X), 200)
_dist, _idx = NearestNeighbors(n_neighbors=_k).fit(X).kneighbors(X)
_cum = np.cumsum(row_sessions[_idx], axis=1)              # sessions within each neighbour distance
_reach = (_cum >= MIN_SESSIONS).argmax(axis=1)
core_dist = np.where(_cum[:, -1] >= MIN_SESSIONS, _dist[np.arange(len(X)), _reach], _dist[:, -1])

eps = EPS if EPS is not None else float(np.percentile(core_dist, EPS_PCT))
eps = max(eps, 1e-6)
_s = np.sort(core_dist)
plt.figure(figsize=(10, 4))
plt.plot(_s, color="#2a78d6")
plt.axhline(eps, color="#c0392b", lw=1)
plt.text(0, eps, f"  EPS = {eps:.3f}" + ("" if EPS is not None else f" ({EPS_PCT}th percentile)"),
         va="bottom", color="#c0392b")
plt.xlabel("distinct rows, sorted"); plt.ylabel("core distance")
plt.title(f"Core distance for MIN_SESSIONS = {MIN_SESSIONS}: pick EPS near where the curve turns up")
plt.show()
print(f"EPS = {eps:.3f}; {int((core_dist > eps).sum()):,} of {len(X):,} distinct rows have a core distance above it")
if _low := {f: w for f in FLAGS if (w := WEIGHTS.get(f, FLAG_WEIGHT)) <= eps}:
    print(f"!! flag weight(s) {_low} not above EPS -- DBSCAN can join their values into one cluster")


In [ ]:
# 7 - DBSCAN on the distinct rows, weighted by sessions; every session gets its row's cluster
# (-1 = noise: the session fits no cluster)
_t0 = time.perf_counter()
db = DBSCAN(eps=eps, min_samples=MIN_SESSIONS, n_jobs=-1).fit(X, sample_weight=row_sessions)
rows["cluster"] = db.labels_
flows["cluster"] = db.labels_[flows_row]
for _c in NUM_COLS:                                        # rounded log values, for the tables
    flows[_c] = feat[_c].to_numpy()

_noise = flows["cluster"].eq(-1)
print(f"DBSCAN in {time.perf_counter() - _t0:.1f}s: {rows['cluster'].max() + 1:,} clusters; noise: "
      f"{int(_noise.sum()):,} sessions ({_noise.mean():.3%}) in {int((rows['cluster'] == -1).sum()):,} distinct rows")


In [ ]:
# 8 - what each cluster is: sessions, and medians of the raw numbers; noise (-1) last
flows["duration"] = (flows["end"] - flows["start"]).astype(float)
flows["bytes_per_packet"] = flows["bytes"] / flows["packets"].where(flows["packets"] > 0)
_g = flows.groupby("cluster")
cluster_profile = pd.DataFrame({
    "sessions":         _g.size(),
    "share":            _g.size() / len(flows),
    "bytes":            _g["bytes"].median(),
    "packets":          _g["packets"].median(),
    "duration":         _g["duration"].median(),
    "bytes_per_packet": _g["bytes_per_packet"].median(),
    "reject":           _g["action"].apply(lambda s: s.eq("REJECT").mean()),
    "north_south":      _g["tag-direction"].apply(lambda s: s.eq("north-south").mean()),
})
cluster_profile = cluster_profile.sort_index(key=lambda c: c.where(c != -1, c.max() + 1))
cluster_profile.style.format({"sessions": "{:,}", "share": "{:.2%}", "bytes": "{:,.0f}",
                              "packets": "{:,.0f}", "duration": "{:,.0f}",
                              "bytes_per_packet": "{:,.0f}", "reject": "{:.0%}",
                              "north_south": "{:.0%}"})


In [ ]:
# 9 - picture: the distinct rows in 3D (PCA of the scaled features), size by sessions (log);
# clusters gray, noise red
_p = PCA(3).fit(X, ).transform(X)
_noiseRow = rows["cluster"].to_numpy() == -1
_size = 6 + 14 * np.log10(row_sessions)
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*_p[~_noiseRow].T, s=_size[~_noiseRow], c="#9a9893", alpha=0.35, depthshade=False)
ax.scatter(*_p[_noiseRow].T, s=_size[_noiseRow] + 10, c="#c0392b", alpha=0.9, depthshade=False)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.set_zlabel("PC 3")
ax.set_title(f"{rows['cluster'].max() + 1:,} clusters (gray), {int(_noiseRow.sum()):,} noise rows (red) -- "
             f"{len(rows):,} distinct rows for {len(flows):,} sessions", fontsize=13, weight="bold")
plt.show()


In [ ]:
# 10 - the output of this notebook, dbscan_output: the noise sessions (cluster -1) as their
# original rows from raw_dbscan (cell 3) -- every field as loaded, unaltered -- with what the
# model adds joined on session_id: cluster, and the two features DBSCAN computed that aren't
# in the raw data, duration (end - start, whole seconds) and bytes_per_packet (rounded to whole bytes). A "type" field first
# says which detection made the rows. Biggest transfers first.

DBSCAN_TYPE = "dbscan anomaly detection"

_noise = flows.loc[flows["cluster"].eq(-1), ["session_id", "cluster"]]
dbscan_output = raw_dbscan[raw_dbscan["session_id"].isin(_noise["session_id"])]
dbscan_output = dbscan_output.merge(_noise, on="session_id", how="left", validate="one_to_one")
dbscan_output["duration"] = (dbscan_output["end"] - dbscan_output["start"]).clip(lower=0).astype("Int64")
dbscan_output["bytes_per_packet"] = (dbscan_output["bytes"]
                                     / dbscan_output["packets"].where(dbscan_output["packets"] > 0)
                                     ).round().astype("Int64")
# IDs as text: an account id can start with 0, which a number would drop
dbscan_output[["account-id", "interface-id", "session_id"]] = (
    dbscan_output[["account-id", "interface-id", "session_id"]].astype("string"))
dbscan_output = dbscan_output.sort_values(["bytes", "packets"], ascending=False).reset_index(drop=True)
dbscan_output.insert(0, "type", DBSCAN_TYPE)
if len(dbscan_output) != len(_noise):
    raise AssertionError(f"{len(_noise):,} noise sessions but {len(dbscan_output):,} output rows")
print(f"{len(dbscan_output):,} flagged sessions, {dbscan_output.shape[1]} fields")
pd.set_option("display.max_rows", 50)

In [ ]:
# 12 - per ENI, the outliers (dbscan_output: cluster -1) as one table of the unique
# combinations of ENI_COMBO_FIELDS, with how many sessions each covers, the total bytes, packets and duration (seconds,
# summed over the sessions, so overlapping sessions can add up to more than the wall-clock
# time) of their sessions, and bytes per packet (total bytes / total packets) -- the features
# DBSCAN scored them on. ENI_SORT picks the column to sort by, largest first. Blank values
# show as "(none)". Edit the list to change the fields. Tables are kept in dbscan_eni_combos.
# ENI_EXCLUDE_REJECT = True leaves out sessions whose action is REJECT (blocked by the
# security group / NACL), so the tables show only traffic that got through.

ENI_COMBO_FIELDS   = ["action", "tag-direction", "tag-service", "tag-asn"]
ENI_EXCLUDE_REJECT = True
ENI_SORT           = "bytes"     # or "sessions", "packets", "duration", "bytes_per_packet"

_vals = ["bytes", "packets", "duration", "bytes_per_packet"]
_fmt  = {"sessions": "{:,}", "bytes": "{:,.0f}", "packets": "{:,.0f}", "duration": "{:,.0f}", "bytes_per_packet": "{:,.0f}"}
_need = ["interface-id"] + _vals + ENI_COMBO_FIELDS
if _miss := [c for c in _need if c not in dbscan_output.columns]:
    raise KeyError(f"not in dbscan_output: {_miss}")
if ENI_SORT not in ["sessions"] + _vals:
    raise ValueError(f"ENI_SORT must be one of {_vals}, not {ENI_SORT!r}")

_src = dbscan_output
if ENI_EXCLUDE_REJECT:
    _rej = _src["action"].eq("REJECT").fillna(False).to_numpy(bool)
    print(f"left out {int(_rej.sum()):,} REJECT sessions, {int((~_rej).sum()):,} left")
    _src = _src[~_rej]
_ec = _src[_need].copy()
_ec[["interface-id"] + ENI_COMBO_FIELDS] = (_ec[["interface-id"] + ENI_COMBO_FIELDS]
                                            .astype("string").fillna("(none)"))
dbscan_eni_combos = {}
for _eni, _d in sorted(_ec.groupby("interface-id", sort=False), key=lambda kv: -len(kv[1])):
    _g = _d.groupby(ENI_COMBO_FIELDS)
    _t = _g[["bytes", "packets", "duration"]].sum()
    _t.insert(0, "sessions", _g.size())
    _t["bytes_per_packet"] = _t["bytes"] / _t["packets"].where(_t["packets"] > 0)
    _t = _t.sort_values(ENI_SORT, ascending=False).reset_index()
    dbscan_eni_combos[_eni] = _t
    print()
    print(f"{_eni}: {len(_t):,} unique combinations in {len(_d):,} outlier sessions -- "
          f"totals, by {ENI_SORT}")
    display(_t.style.format(_fmt))


In [ ]:
# 13 - connection diagram per ENI for the outliers (dbscan_output: cluster -1), as pyod-vflows'
# cell 27b: who each ENI's outlier sessions talked to, one figure each.
#   middle -- the ENI (violet circle), labelled with its own address
#   left   -- its east-west peers: another ENI (violet circle -- lateral movement shows up
#             here), or a private address that isn't an ENI in these logs (gray diamond:
#             another VPC, an AWS service, 100.64/10, ...)
#   right  -- the ASNs (tag-asn) of its north-south peers (blue squares), busiest at the top,
#             names longer than EG_WRAP characters wrapped. Hover one for its countries. The
#             top EG_TOP_N ASNs get their own node, the rest share one "other" node.
# Line width grows with sessions (log scale). Every line is labelled with its tag-service
# -- the most common one, "+n" when there are more; a session with no tag-service counts as
# "port N" (its lower port). EG_SERVICE_LABELS = False hides the labels. An ASN line's label
# also gives the total bytes and duration of its sessions (duration summed over the sessions,
# so overlapping ones can add up to more than the elapsed time; EG_LINE_STATS = False hides
# them). Hover a line's dot for sessions, total bytes and duration, the top 8
# services with their counts, top ports and first / last seen.
# An ENI's own address is the one present in every one of its records in flows (all loaded
# sessions), so a peer ENI is named even when it has no outlier sessions itself.
# EG_SOURCE picks the sessions: dbscan_output (default) or e.g. flows, to see every link.
# EG_ENIS limits it to some ENIs (None = all). EG_EXCLUDE_REJECT = True leaves out REJECT
# sessions (follows ENI_EXCLUDE_REJECT from cell 12). Figures are kept in dbscan_eni_graphs.

EG_SOURCE         = dbscan_output
EG_ENIS           = None          # e.g. ["eni-02b85e2a3caf5dd60", "eni-0f324b6f6044e4ef3"]
EG_TOP_N          = 25
EG_SERVICE_LABELS = True
EG_LINE_STATS     = True          # total bytes and duration on each ASN line
EG_WRAP           = 40            # wrap ASN names longer than this many characters
EG_EXCLUDE_REJECT = globals().get("ENI_EXCLUDE_REJECT", True)

_EG_ENI, _EG_CTY, _EG_OTH = "#4a3aa7", "#2a78d6", "#898781"   # violet, blue, gray
_EG_NS_LINE, _EG_EW_LINE = "rgba(42,120,214,0.45)", "rgba(74,58,167,0.85)"
_EG_INK, _EG_INK2, _EG_SURF = "#0b0b0b", "#52514e", "#fcfcfb"

_need = ["interface-id", "srcaddr", "dstaddr", "srcport", "dstport", "tag-direction",
         "tag-country", "tag-asn", "tag-service", "action", "bytes", "duration", "start_time"]
if _miss := [c for c in _need if c not in EG_SOURCE.columns]:
    raise KeyError(f"not in EG_SOURCE: {_miss}")

# each ENI's own address: of its first record's two addresses, the one in all its records
_r   = flows.loc[flows["srcaddr"].ne("-") & flows["srcaddr"].notna(),
                    ["interface-id", "srcaddr", "dstaddr"]]
_fst = _r.groupby("interface-id", sort=False)[["srcaddr", "dstaddr"]].first()
_ipEni, _eniIp = {}, {}
for _side in ("srcaddr", "dstaddr"):
    _cand = _r["interface-id"].map(_fst[_side])
    _cov  = (_r["srcaddr"].eq(_cand) | _r["dstaddr"].eq(_cand)).groupby(_r["interface-id"]).mean()
    for _eni, _share in _cov.items():
        if _share == 1.0 and _eni not in _eniIp:
            _eniIp[_eni] = _fst.at[_eni, _side]
            _ipEni.setdefault(_eniIp[_eni], _eni)
if _noOwn := sorted(set(_r["interface-id"].unique()) - set(_eniIp)):
    print(f"!! no single own address for {len(_noOwn)} ENIs (e.g. {_noOwn[0]}) -- "
          "their east-west peers show as bare addresses")

_hasRank = "rank" in EG_SOURCE.columns
_s = EG_SOURCE[_need + (["rank"] if _hasRank else [])]
if EG_ENIS:
    _s = _s[_s["interface-id"].isin(EG_ENIS)]
if EG_EXCLUDE_REJECT:
    _rej = _s["action"].eq("REJECT").fillna(False).to_numpy(bool)
    print(f"left out {int(_rej.sum()):,} REJECT sessions, {int((~_rej).sum()):,} left")
    _s = _s[~_rej]
_s = _s.assign(_eni=_s["interface-id"].astype("string"),
               _dir=_s["tag-direction"].astype("string"),
               _svc=_s["tag-service"].astype("string"),
               _port=_s["dstport"].where(_s["dstport"].le(_s["srcport"]) | _s["srcport"].isna(),
                                         _s["srcport"]).astype("Int64").astype("string").fillna("-"))
_s["_svc"] = _s["_svc"].fillna("port " + _s["_port"])


def _egFmtBytes(b):
    for u in ("B", "kB", "MB", "GB", "TB"):
        if abs(b) < 1000 or u == "TB":
            return f"{b:,.0f} {u}" if u == "B" else f"{b:,.1f} {u}"
        b /= 1000


def _egFmtDur(sec):
    """59 -> '59 s', 10587 -> '2.9 h', 366162 -> '4.2 d'."""
    if pd.isna(sec):
        return "-"
    for lim, div, u in ((120, 1, "s"), (7200, 60, "min"), (172800, 3600, "h")):
        if sec < lim:
            return f"{sec / div:,.0f} {u}" if u == "s" else f"{sec / div:,.1f} {u}"
    return f"{sec / 86400:,.1f} d"


def _egRank(r):
    """', rank N' for a row of an edge table, '' when there is no rank."""
    v = r.get("rank") if _hasRank else None
    return f",pyod rank {int(v)}" if v is not None and pd.notna(v) else ""


def _egEdges(d, key):
    """One row per value of key (the far end): sessions, bytes, highest rank, services, top
    ports, first / last seen; most sessions first."""
    g = d.assign(_k=key.to_numpy())
    _agg = dict(sessions=("_k", "size"), bytes=("bytes", "sum"),
                dur=("duration", "sum"),
                first=("start_time", "min"), last=("start_time", "max"))
    if _hasRank:
        _agg["rank"] = ("rank", "max")
    t = g.groupby("_k").agg(**_agg)
    t["countries"] = g.groupby("_k")["tag-country"].agg(
        lambda x: ", ".join(x.astype("string").fillna("(none)").value_counts().index[:5]))
    t["ports"] = g.groupby("_k")["_port"].agg(lambda x: ", ".join(x.value_counts().index[:3]))
    # services on each line, most sessions first: "SSH 120, RDP 4" and the label "SSH +1"
    _sv = (g.groupby(["_k", "_svc"]).size().rename("n").reset_index()
           .sort_values(["_k", "n"], ascending=[True, False]))
    _sv["_txt"] = _sv["_svc"] + " " + _sv["n"].map("{:,}".format)
    _grp = _sv.groupby("_k", sort=False)
    t["services"]  = _grp["_txt"].agg(lambda x: ", ".join(x.iloc[:8])
                                      + (f", ... {len(x) - 8:,} more" if len(x) > 8 else ""))
    t["svc_label"] = _grp["_svc"].agg(lambda x: x.iloc[0] + (f" +{len(x) - 1}" if len(x) > 1 else ""))
    return t.sort_values(["sessions", "bytes"], ascending=False)


def _egFold(t, n, what):
    """Keep the top n rows; the rest become one 'other (k <what>)' row."""
    if len(t) <= n:
        return t
    rest = t.iloc[n:]
    other = pd.DataFrame({"sessions": [rest["sessions"].sum()], "bytes": [rest["bytes"].sum()], "dur": [rest["dur"].sum()],
                          "first": [rest["first"].min()], "last": [rest["last"].max()],
                          "countries": ["-"],
                          "ports": ["-"], "services": ["-"], "svc_label": [""]},
                         index=[f"other ({len(rest)} {what})"])
    if "rank" in t.columns:
        other["rank"] = rest["rank"].max()
    return pd.concat([t.iloc[:n], other])


def _egN(n, word="session"):
    return f"{n:,} {word}" + ("" if n == 1 else "s")


def _egGraph(eni, d):
    own  = _eniIp.get(eni)
    ns   = d[d["_dir"].eq("north-south").fillna(False)]
    ew   = d[d["_dir"].eq("east-west").fillna(False)]
    asnT = _egFold(_egEdges(ns, ns["tag-asn"].astype("string").fillna("(none)")),
                   EG_TOP_N, "ASNs")
    peer = ew["dstaddr"].where(ew["srcaddr"].eq(own), ew["srcaddr"]).astype("string")
    peerT = _egFold(_egEdges(ew, peer.map(lambda a: _ipEni.get(a, a))), EG_TOP_N, "peers")

    span = max(len(asnT), len(peerT), 2) - 1
    def spread(n):                                   # n slots from top (0) to bottom (-span)
        return [-span / 2] if n == 1 else [-i * span / (n - 1) for i in range(n)]
    hub  = (0.0, -span / 2)
    maxS = max([1] + asnT["sessions"].tolist() + peerT["sessions"].tolist())
    width = lambda n: 1 + 5 * np.log10(n) / max(np.log10(maxS), 1)

    fig = go.Figure()
    mid = {"x": [], "y": [], "text": [], "color": [], "label": []}
    for t, x, line, dot in ((peerT, -1.0, _EG_EW_LINE, _EG_ENI), (asnT, 1.0, _EG_NS_LINE, _EG_CTY)):
        for (name, r), y in zip(t.iterrows(), spread(len(t))):
            fig.add_scatter(x=[hub[0], x], y=[hub[1], y], mode="lines", hoverinfo="skip",
                            showlegend=False, line=dict(color=line, width=width(r.sessions)))
            # dot (hover target + label) 70% of the way out, where the lines have spread apart
            mid["x"].append(hub[0] + 0.7 * (x - hub[0])); mid["y"].append(hub[1] + 0.7 * (y - hub[1]))
            mid["color"].append(dot)
            _lbl = [r.svc_label] if EG_SERVICE_LABELS and r.svc_label else []
            if EG_LINE_STATS and t is asnT:
                _lbl.append(f"{_egFmtBytes(r.bytes)}, {_egFmtDur(r.dur)}")
            mid["label"].append("<br>".join(_lbl))
            mid["text"].append(f"<b>{eni}  &harr;  {name}</b><br>{_egN(int(r.sessions))}, "
                               f"{_egFmtBytes(r.bytes)}{_egRank(r)}"
                               + f", {_egFmtDur(r.dur)}"
                               + f"<br>services {r.services}<br>ports {r.ports}"
                               f"<br>first {str(r['first'])[:19]}<br>last  {str(r['last'])[:19]}")
    fig.add_scatter(x=mid["x"], y=mid["y"], mode="markers+text", showlegend=False,
                    text=mid["label"], textposition="top center",
                    textfont=dict(size=10, color=_EG_INK2),
                    marker=dict(size=6, color=mid["color"], line=dict(width=1.5, color=_EG_SURF)),
                    hovertext=mid["text"], hoverinfo="text")

    allLbl = {"left": [], "right": []}            # labels per side, to size margins / rows
    def nodes(names, ys, x, color, symbol, side, legend, label, hover=None):
        if names:
            if x:
                allLbl["left" if x < 0 else "right"] += [label(n) for n in names]
            fig.add_scatter(x=[x] * len(names), y=ys, mode="markers+text", name=legend,
                            text=[label(n) for n in names], textposition=side,
                            textfont=dict(color=_EG_INK, size=12), hoverinfo="text",
                            cliponaxis=False,
                            hovertext=[hover(n) if hover else f"<b>{label(n)}</b>" for n in names],
                            marker=dict(size=14, color=color, symbol=symbol,
                                        line=dict(width=2, color=_EG_SURF)))

    _asnN = lambda a: f"({int(asnT.at[a, 'sessions']):,}{_egRank(asnT.loc[a])})"
    nodes(list(asnT.index), spread(len(asnT)), 1.0, _EG_CTY, "square", "middle right", "ASN",
          lambda a: "<br>".join(textwrap.wrap(a, EG_WRAP) or [a]) + f"  {_asnN(a)}",
          lambda a: f"<b>{a}</b>  {_asnN(a)}<br>countries: {asnT.at[a, 'countries']}")
    _isEni = [p in _eniIp for p in peerT.index]
    _py = spread(len(peerT))
    for flag, color, symbol, legend in ((True, _EG_ENI, "circle", "peer ENI"),
                                        (False, _EG_OTH, "diamond", "other private address")):
        _n = [p for p, f in zip(peerT.index, _isEni) if f == flag]
        nodes(_n, [y for y, f in zip(_py, _isEni) if f == flag], -1.0, color, symbol, "middle left",
              legend, lambda p: "<br>".join(x for x in (p, _eniIp.get(p, ""),
                  f"({int(peerT.at[p, 'sessions']):,}{_egRank(peerT.loc[p])})") if x))
    nodes([eni], [hub[1]], 0.0, _EG_ENI, "circle", "bottom center", "this ENI",
          lambda e: f"<b>{e}</b><br>{own or ''}")
    fig.data[-1].marker.size = 22

    if not len(asnT):
        allLbl["right"].append("no north-south sessions")
        fig.add_annotation(x=1.0, y=hub[1], text="no north-south sessions", showarrow=False,
                           font=dict(color=_EG_INK2), xanchor="left")
    if not len(peerT):
        allLbl["left"].append("no east-west sessions")
        fig.add_annotation(x=-1.0, y=hub[1], text="no east-west sessions", showarrow=False,
                           font=dict(color=_EG_INK2), xanchor="right")
    # label sizes: widest line per side sets that margin (~7.5 px per character at size 12),
    # most lines in one label sets the row height (~16 px per line)
    def _lines(lbls):
        return [re.sub(r"<[^>]+>", "", l) for x in lbls for l in x.split("<br>")]
    _ml = lambda side: max([len(l) for l in _lines(allLbl[side])] + [0])
    _maxLn = max([x.count("<br>") + 1 for v in allLbl.values() for x in v] + [1])
    _row = max(30, 16 * _maxLn + 8)
    h = max(300, _row * (span + 1) + 150)
    fig.update_layout(height=h, paper_bgcolor=_EG_SURF, plot_bgcolor=_EG_SURF,
                      margin=dict(l=int(30 + 7.5 * _ml("left")), r=int(30 + 7.5 * _ml("right")),
                                  t=80, b=20),
                      font=dict(family="system-ui, sans-serif", size=12, color=_EG_INK),
                      hoverlabel=dict(bgcolor=_EG_SURF, font_color=_EG_INK),
                      legend=dict(orientation="h", x=0.01, y=1.0, yanchor="bottom", font_color=_EG_INK2),
                      title=dict(text=f"{eni}  {own or ''}: {_egN(len(ew))} east-west, "
                                      f"{_egN(len(ns))} north-south", x=0.01, xref="container",
                                 xanchor="left", font=dict(size=15, color=_EG_INK)))
    fig.update_xaxes(visible=False, range=[-1.08, 1.08])   # labels sit in the margins
    fig.update_yaxes(visible=False, range=[-span - 0.9, 0.7])
    return fig


dbscan_eni_graphs = {}
for _eni, _d in sorted(_s.groupby("_eni"), key=lambda kv: -len(kv[1])):    # busiest ENI first
    dbscan_eni_graphs[_eni] = _egGraph(_eni, _d)
    dbscan_eni_graphs[_eni].show()

In [ ]:
# 14 - write dbscan_output (cell 10) to parquet and CSV, named after the AWS account and today's date,
# e.g. dbscan-output-637602092272-november-10-2026.parquet (and .csv). The account comes from the
# account-id of the loaded sessions (flows); several accounts are joined with "+", as in the
# pre-processor's file names. OUT_DIR is the folder it goes in.
OUT_DIR = "output"

if "account-id" not in flows.columns:
    raise KeyError("flows has no 'account-id' column -- can't name the file")
_accts = sorted(flows["account-id"].dropna().astype(str).str.strip().unique())
_accts = [a for a in _accts if a and a != "-"]
if not _accts:
    raise ValueError("no account-id values in flows -- can't name the file")
_now = pd.Timestamp.now()
_base = os.path.join(
    OUT_DIR, f"dbscan-output-{'+'.join(_accts)}-{_now.strftime('%B').lower()}-{_now.day}-{_now.year}")
os.makedirs(OUT_DIR, exist_ok=True)
DBSCAN_OUT_PATH = _base + ".parquet"
DBSCAN_OUT_CSV  = _base + ".csv"
dbscan_output.to_parquet(DBSCAN_OUT_PATH, index=False)
dbscan_output.to_csv(DBSCAN_OUT_CSV, index=False)
print(f"wrote {len(dbscan_output):,} rows, {dbscan_output.shape[1]} columns to")
print(f"  {DBSCAN_OUT_PATH}")
print(f"  {DBSCAN_OUT_CSV}")
